# TaskRegistry Prompt Suite Demo

Use this notebook when you only want to register labeled synthetic prompts, run suites, and compare runs by label.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "packages" / "research").exists() and (candidate / "packages" / "pipeline").exists():
        ROOT = candidate
        break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from packages.research.experiment import (
    ExperimentRunner,
    TaskRegistry,
    TestPrompt,
    compare_runs,
    compare_runs_by_prompt_label,
    format_comparison_table,
    format_grouped_comparison_tables,
)


## Register Prompts

The label values are fixed so comparisons stay clean across experiments.


In [ ]:
runner = ExperimentRunner()
registry = TaskRegistry(store=runner.store)

prompts: list[TestPrompt] = [
    {"prompt": "hexapod robot with explicitly provided six-legged morphology", "label": "morphlogy-provided"},
    {"prompt": "robot that should infer one body plan for stair climbing", "label": "infer-morphology-single"},
    {"prompt": "robot family for tunnels, rubble, and uneven terrain", "label": "infer-morphology-family"},
    {"prompt": "invent a robot for moving tools around a repair bench", "label": "open-ended"},
]

registry.register_many(prompts)
registry.list()


## Run Every Prompt Against One or More Strategies

Use any registered strategy. Start with one strategy, then add more to compare them under each prompt label.


In [ ]:
experiment_name = "task-registry-demo"

runs = runner.run_suite(
    test_prompts=prompts,
    strategy_names=["grammar"],
    experiment_name=experiment_name,
    seed=42,
    max_candidates=3,
    extra={"max_attempts": 2},
)

[(run.run_id, run.prompt_label, run.strategy_name, run.error) for run in runs]


## Compare: Label-Agnostic and Grouped

The flat table is still useful for all-run sorting. The grouped table is better for prompt-suite benchmarking.


In [ ]:
exp = runner.store.find_experiment_by_name(experiment_name)
all_runs = runner.store.list_runs(exp.experiment_id) if exp else []

print("Flat:")
print(format_comparison_table(compare_runs(all_runs, runner.store), include_label=True))

print("\nGrouped by prompt label:")
print(format_grouped_comparison_tables(compare_runs_by_prompt_label(all_runs, runner.store)))
